# 🗑️ EcoClean Civic — YOLOv8 Waste Detection Training

This notebook trains a custom YOLOv8s model on the **TACO dataset** (Trash Annotations in Context),
remapping its 60 categories into the 9 waste classes used by the EcoClean backend.

**Runtime:** ~30–45 min on a free Colab T4 GPU  
**Output:** `best.pt` saved to your Google Drive at `/MyDrive/waste_model/best.pt`

### Steps
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`
2. Run all cells top-to-bottom
3. After training, copy `best.pt` path to `WASTE_MODEL_WEIGHTS=` in your `.env`

> **Security note:** Your Roboflow API key is entered via a password prompt and is
> never stored in this notebook or committed to git.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## Step 2 — Mount Google Drive (output will be saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/waste_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Output directory: {OUTPUT_DIR}')

## Step 3 — Download TACO Dataset via Roboflow

In [ ]:
import getpass
from roboflow import Roboflow

# Enter your Roboflow API key when prompted (never stored in the notebook)
RF_API_KEY = getpass.getpass('Roboflow API key: ')

rf = Roboflow(api_key=RF_API_KEY)

# TACO dataset on Roboflow (public, YOLO format)
project = rf.workspace('bedlopie').project('taco-trash-annotations-in-context')
version = project.version(1)
dataset = version.download('yolov8', location='/content/taco_raw')

print(f'✅ Dataset downloaded to: {dataset.location}')

## Step 4 — Remap 60 TACO Classes → 9 EcoClean Classes

In [ ]:
import os, shutil, yaml
from pathlib import Path

# ── Mapping: TACO label (lowercase) → EcoClean class index ──────────────────
ECOCLEAN_CLASSES = [
    'plastic', 'paper', 'cardboard', 'glass', 'metal',
    'organic', 'hazardous', 'construction_debris', 'mixed_litter'
]

TACO_TO_ECOCLEAN = {
    # plastic
    'plastic bag': 0, 'plastic bottle': 0, 'plastic film': 0,
    'single-use carrier bag': 0, 'polystyrene item': 0, 'plastic straw': 0,
    'plastic lid': 0, 'plastic utensils': 0, 'plastic container': 0,
    'six pack rings': 0, 'plastic cup': 0,
    # paper
    'paper': 1, 'newspaper': 1, 'paper bag': 1, 'paper cup': 1,
    'paper straw': 1, 'tissue': 1, 'wrapping paper': 1,
    # cardboard
    'cardboard': 2, 'corrugated cardboard': 2, 'pizza box': 2,
    # glass
    'glass bottle': 3, 'glass jar': 3, 'broken glass': 7,
    # metal
    'aluminium can': 4, 'aluminium foil': 4, 'metal bottle cap': 4,
    'steel can': 4, 'aerosol': 4, 'tin can': 4,
    # organic
    'food waste': 5, 'organic': 5, 'banana peel': 5, 'other food': 5,
    # hazardous
    'battery': 6, 'cigarette': 6, 'lighter': 6, 'medication': 6,
    'syringe': 6, 'e-waste': 6,
    # construction_debris
    'rope': 7, 'wire': 7, 'foam': 7, 'construction debris': 7,
    # mixed_litter (catch-all)
    'unlabeled litter': 8, 'shoe': 8, 'clothing': 8, 'other': 8,
}

DEFAULT_CLASS = 8  # mixed_litter for anything unmapped

# ── Read original data.yaml to get class names ──────────────────────────────
raw_yaml_path = Path(dataset.location) / 'data.yaml'
with open(raw_yaml_path) as f:
    raw_cfg = yaml.safe_load(f)

original_names = raw_cfg.get('names', [])
print(f'Original TACO classes ({len(original_names)}): {original_names[:10]} ...')

# Build per-original-id → ecoclean-id map
id_map = {}
for i, name in enumerate(original_names):
    key = name.lower().strip()
    id_map[i] = TACO_TO_ECOCLEAN.get(key, DEFAULT_CLASS)

print('\n📦 Class remapping sample:')
for i, name in list(enumerate(original_names))[:10]:
    print(f'  {i:3d} "{name}" → {id_map[i]} ({ECOCLEAN_CLASSES[id_map[i]]})')

In [ ]:
# ── Rewrite all .txt label files with remapped class IDs ────────────────────
RAW_DIR = Path(dataset.location)
REMAPPED_DIR = Path('/content/taco_remapped')

for split in ['train', 'valid', 'test']:
    src_img = RAW_DIR / split / 'images'
    src_lbl = RAW_DIR / split / 'labels'
    if not src_lbl.exists():
        continue

    dst_img = REMAPPED_DIR / split / 'images'
    dst_lbl = REMAPPED_DIR / split / 'labels'
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    # Copy images
    for img in src_img.glob('*'):
        shutil.copy2(img, dst_img / img.name)

    # Remap labels
    for lbl in src_lbl.glob('*.txt'):
        new_lines = []
        for line in lbl.read_text().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            orig_cls = int(parts[0])
            new_cls = id_map.get(orig_cls, DEFAULT_CLASS)
            new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
        (dst_lbl / lbl.name).write_text('\n'.join(new_lines))

    print(f'✅ {split}: {len(list(src_lbl.glob("*.txt")))} label files remapped')

# ── Write new data.yaml ──────────────────────────────────────────────────────
new_yaml = {
    'path': str(REMAPPED_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(ECOCLEAN_CLASSES),
    'names': ECOCLEAN_CLASSES,
}
yaml_path = REMAPPED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(new_yaml, f, default_flow_style=False)

print(f'\n✅ data.yaml written → {yaml_path}')
print(yaml.dump(new_yaml))

## Step 5 — Train YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # small variant — good balance of speed/accuracy

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    patience=20,          # early stopping
    project='/content/runs',
    name='waste_yolov8s',
    exist_ok=True,
    device=0,             # GPU
)

print('\n🏁 Training complete!')
print(f'Best weights: {results.save_dir}/weights/best.pt')

## Step 6 — Validate & Print Metrics

In [ ]:
best_pt = f'{results.save_dir}/weights/best.pt'
trained_model = YOLO(best_pt)

metrics = trained_model.val(data=str(yaml_path), device=0)

print('\n📊 Validation Metrics')
print(f'  mAP50      : {metrics.box.map50:.4f}')
print(f'  mAP50-95   : {metrics.box.map:.4f}')
print(f'  Precision  : {metrics.box.mp:.4f}')
print(f'  Recall     : {metrics.box.mr:.4f}')
print(f'\nClass-wise mAP50:')
for i, cls_name in enumerate(ECOCLEAN_CLASSES):
    print(f'  {cls_name:<22}: {metrics.box.ap50[i]:.4f}')

## Step 7 — Export best.pt to Google Drive

In [ ]:
import shutil
from pathlib import Path

src = Path(best_pt)
dst = Path(OUTPUT_DIR) / 'best.pt'
shutil.copy2(src, dst)

print(f'✅ Model exported to Google Drive: {dst}')
print()
print('━' * 60)
print('NEXT STEP — add this to your .env file:')
print(f'  WASTE_MODEL_WEIGHTS=/path/to/best.pt')
print()
print('After downloading best.pt from Drive, set the full local path.')
print('The Flask app will auto-switch from mock → real inference.')
print('━' * 60)

## Optional — Quick Inference Test
Upload a test image to verify the model works before deploying.

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

print('Upload a waste image to test inference...')
uploaded = files.upload()

for fname in uploaded:
    result = trained_model(fname)[0]
    img_annotated = result.plot()
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_annotated[:, :, ::-1])
    plt.axis('off')
    plt.title('EcoClean Waste Detection — YOLOv8s')
    plt.tight_layout()
    plt.show()
    
    print(f'\nDetections in {fname}:')
    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        print(f'  {ECOCLEAN_CLASSES[cls_id]:<22} conf={conf:.2f}')